<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/snes/notebooks/06_sales_merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
from io import StringIO
import requests

# ==============================================================================
# PIPELINE CONFIGURATION
# ==============================================================================
CONSOLE_NAME = "snes"
WIKI_SALES_URL = "https://en.wikipedia.org/wiki/List_of_best-selling_Super_Nintendo_Entertainment_System_video_games"

DATASETS_DIR = "datasets"
# ADJUSTMENT: Correct chained sequence tracking for Step 06
INPUT_FILENAME = f"5_{CONSOLE_NAME}_games_with_genres.xlsx"    # Reading from Code 5
OUTPUT_FILENAME = f"6_{CONSOLE_NAME}_games_perfect_list.xlsx"   # Generating final base number 6

INPUT_PATH = os.path.join(DATASETS_DIR, INPUT_FILENAME)
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def pipeline_final_sales_gold(input_path, output_path, sales_url):
    """
    Scrapes best-selling metrics from Wikipedia and marries them with the analytical database.
    Acts as the Ultimate Gold Layer (The Perfect List) of the Medallion Architecture.
    """
    print(f"🏆 [CODE 06] Starting Pipeline Closure (Final Gold Layer): {CONSOLE_NAME.upper()}")

    # 1. Pipeline gatekeeper check
    if not os.path.exists(input_path):
        print(f"❌ Critical Error: Input file '{input_path}' not found. Please run Notebook 05 first.")
        return None

    df_games = pd.read_excel(input_path)
    print(f"📊 Dataset with scores and categories loaded: {len(df_games)} records.")

    # 2. Extract raw finance/sales data from Wikipedia
    print("🌐 Fetching sales volume metrics from Wikipedia...")
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(sales_url, headers=headers)
        response.raise_for_status()

        html_data = StringIO(response.text)
        tables = pd.read_html(html_data)

        # Target the biggest financial table containing global milestone data
        df_sales = max(tables, key=len)
        print("✅ Best-sellers metrics downloaded successfully.")
    except Exception as e:
        print(f"❌ Error connecting to Wikipedia sales stream: {e}")
        return None

    # Cleans bracket references out of column names
    df_sales.columns = [col.split('[')[0].strip() for col in df_sales.columns]

    # Dynamically maps column headers based on semantic syntax
    col_game = [c for c in df_sales.columns if 'Game' in c or 'Title' in c][0]
    col_sales = [c for c in df_sales.columns if 'Sales' in c or 'Copies' in c][0]

    df_sales_clean = df_sales[[col_game, col_sales]].copy()
    df_sales_clean.columns = ['Match_Title', 'Copies_Sold_Millions']

    # Aggressive regex formatting to parse strings like "11.61 million[3]" into clean floats
    df_sales_clean['Copies_Sold_Millions'] = df_sales_clean['Copies_Sold_Millions'].astype(str)\
        .str.replace(r'million.*|\[.*\]', '', regex=True)\
        .str.strip().astype(float)

    print(f"💰 Financial parameters mapped for the top {len(df_sales_clean)} commercial hits.")

    # 3. Data Integration (Smart Left Join)
    # Generates standard normalized alphanumeric keys to bypass formatting discrepancies
    df_games['join_key'] = df_games['T1_Clean'].astype(str).str.lower().str.replace(r'\W+', '', regex=True)
    df_sales_clean['join_key'] = df_sales_clean['Match_Title'].astype(str).str.lower().str.replace(r'\W+', '', regex=True)

    # Eliminates lookup duplicates to preserve the exact row shape of our analytical dataframe
    df_sales_clean = df_sales_clean.drop_duplicates(subset=['join_key'])

    # Merges sales performance metrics into our main engine
    df_final = pd.merge(df_games, df_sales_clean[['join_key', 'Copies_Sold_Millions']], on='join_key', how='left')

    # 4. Handling Missing Values (NaN remediation)
    # Non-record-breaking games default to 0 (indicating sales under the 1 million threshold)
    df_final['Copies_Sold_Millions'] = df_final['Copies_Sold_Millions'].fillna(0)

    # 5. Dropping operational overhead keys and sorting features
    df_final = df_final.drop(columns=['join_key'])

    final_ordered_columns = [
        'Title',
        'T1_Clean',
        'IGDB_Score',
        'Copies_Sold_Millions',
        'Category_Primary',
        'Category_Secondary',
        'Category_Tertiary',
        'Original_Release',
        'Developer',
        'Publisher'
    ]
    existing_columns = [c for c in final_ordered_columns if c in df_final.columns]
    df_final = df_final[existing_columns]

    # 6. Multi-attribute Sorting (Ranks by maximum IGDB score, using sales data as a secondary tie-breaker)
    df_final = df_final.sort_values(by=['IGDB_Score', 'Copies_Sold_Millions'], ascending=[False, False])

    # 7. Final Artifact Persistence
    df_final.to_excel(output_path, index=False)
    print(f"\n🎮 PIPELINE CLOSED SUCCESSFULLY! Ultimate dataset saved at: {output_path}")
    print("═" * 80)

    # Returns final preview to the notebook dashboard
    print(f"\n👑 TOP 5 GAMES ON YOUR PERFECT LIST (MAX SCORES TIED WITH SALES VOLUME):")
    return df_final[['T1_Clean', 'IGDB_Score', 'Copies_Sold_Millions', 'Category_Primary']].head(5)

# Trigger the final core pipeline
df_perfect_list_preview = pipeline_final_sales_gold(INPUT_PATH, OUTPUT_PATH, WIKI_SALES_URL)